# Synapse Data Pipeline no Google Colab

Este notebook executa coleta autorizada, limpeza, deduplicação, validação e publicação no Hugging Face.

> Use somente fontes cuja coleta e redistribuição sejam permitidas. O pipeline respeita `robots.txt`, não contorna autenticação/paywalls/CAPTCHAs e exige metadados de licença. Para escala de bilhões de tokens, use shards, armazenamento persistente e execução distribuída; uma sessão gratuita do Colab não é suficiente.

In [ ]:
# 1) Clonar o pipeline e instalar dependências
!git clone https://github.com/Comunidadesynapsebr/synapse-data-pipeline.git
%cd synapse-data-pipeline
!pip -q install -e .

In [ ]:
# 2) Configurar o pipeline
!cp configs/config.example.yaml configs/config.yaml

# Edite este bloco para definir o dataset, formato e limites.
from pathlib import Path
import yaml

config_path = Path('configs/config.yaml')
config = yaml.safe_load(config_path.read_text())
config['project']['dataset_name'] = 'Comunidade-Synapse-BR/dataset-publico'
config['project']['output_format'] = 'jsonl'  # jsonl | parquet | csv | txt
config['collection']['concurrency'] = 8
config['collection']['requests_per_second'] = 2.0
config['cleaning']['min_chars'] = 200
config_path.write_text(yaml.safe_dump(config, sort_keys=False, allow_unicode=True))
print(config_path.read_text())

In [ ]:
# 3) Guardar o token com segurança
# No Colab: painel esquerdo -> Secrets (ícone de chave) -> adicionar HF_TOKEN.
# Nunca cole o token em uma célula ou no GitHub.
from google.colab import userdata
import os

token = userdata.get('HF_TOKEN')
if not token:
    raise RuntimeError('Adicione HF_TOKEN aos Secrets do Colab antes de publicar.')
os.environ['HF_TOKEN'] = token
print('HF_TOKEN carregado com segurança:', bool(token))

In [ ]:
# 4) Enviar um manifesto JSONL com URLs autorizadas
# Cada linha deve conter url, license e, opcionalmente, title.
from google.colab import files
uploaded = files.upload()
manifest_name = next(iter(uploaded))
!mkdir -p data/raw data/clean
!cp "{manifest_name}" examples/manifest.jsonl
print('Manifesto recebido:', manifest_name)

In [ ]:
# 5) Coleta -> limpeza/deduplicação -> validação
!python -m synapse_pipeline.cli collect --manifest examples/manifest.jsonl --out data/raw/raw.jsonl --config configs/config.yaml
!python -m synapse_pipeline.cli clean --input data/raw/raw.jsonl --out data/clean/dataset.jsonl --config configs/config.yaml
!python -m synapse_pipeline.cli validate --input data/clean/dataset.jsonl --config configs/config.yaml
!wc -l data/raw/raw.jsonl data/clean/dataset.jsonl

In [ ]:
# 6) Publicar no Hugging Face somente após revisar licenças e qualidade
# Remova o comentário da linha abaixo quando a revisão estiver concluída.
# !python -m synapse_pipeline.cli publish --input data/clean/dataset.jsonl --config configs/config.yaml

## Escala de 3 bilhões de tokens

Para esse volume, não use uma única célula/sessão. Divida o corpus em shards JSONL/Parquet, processe cada shard em workers, registre hashes e licenças, compacte os resultados e publique arquivos menores no dataset Hub. O notebook acima é um executor de lote e deve receber fontes licenciadas suficientes; ele não gera conteúdo por conta própria.